In [4]:
import findspark
findspark.init()

In [6]:
from pyspark.sql.session import SparkSession
spark = SparkSession.builder.getOrCreate()

In [9]:
DF = spark.read.csv('airbnb_price_prediction_sample.csv',header=True)

## DataPreprocessing and EDA

In [10]:
DF.printSchema()

root
 |-- id: string (nullable = true)
 |-- bathrooms: string (nullable = true)
 |-- bedrooms: string (nullable = true)
 |-- beds: string (nullable = true)
 |-- accommodates: string (nullable = true)
 |-- minimum_nights: string (nullable = true)
 |-- number_of_reviews: string (nullable = true)
 |-- review_scores_rating: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- neighborhood: string (nullable = true)
 |-- price: string (nullable = true)



In [11]:
DF.show()

+---+---------+--------+----+------------+--------------+-----------------+--------------------+-------------+---------------+---------------+------+
| id|bathrooms|bedrooms|beds|accommodates|minimum_nights|number_of_reviews|review_scores_rating|property_type|      room_type|   neighborhood| price|
+---+---------+--------+----+------------+--------------+-----------------+--------------------+-------------+---------------+---------------+------+
|  1|      1.0|     1.0| 2.0|           3|             2|               57|                74.2|    Apartment|    Shared room|Garden District|146.65|
|  2|      3.0|     1.0| 1.0|           2|             2|               59|                85.2|    Apartment|    Shared room|         Marina| 252.6|
|  3|      3.0|     1.0| 1.0|           4|             3|              207|                96.1|        House|    Shared room|    City Center|274.16|
|  4|      1.0|     1.0| 1.0|           2|             1|               97|                72.9| Pri

In [15]:
DF.select('room_type').distinct().show()

+---------------+
|      room_type|
+---------------+
|    Shared room|
|Entire home/apt|
|   Private room|
+---------------+



In [16]:
DF.select('property_type').distinct().show()

+-------------+
|property_type|
+-------------+
|    Apartment|
|       Studio|
|        Condo|
|        House|
| Private room|
+-------------+



In [17]:
DF.select('neighborhood').distinct().show()

+---------------+
|   neighborhood|
+---------------+
|    City Center|
|       Downtown|
|         Marina|
|Garden District|
|       Old Town|
+---------------+



Converting Dtypes

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType

# Cast a single column
for i in DF.columns[1:8]:
    DF = DF.withColumn(i, col(i).cast(DoubleType()))

In [24]:
DF = DF.withColumn('price', col('price').cast(DoubleType()))

In [25]:
DF.describe().show()

+-------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+--------------------+-------------+---------------+------------+-----------------+
|summary|                id|         bathrooms|          bedrooms|              beds|     accommodates|    minimum_nights| number_of_reviews|review_scores_rating|property_type|      room_type|neighborhood|            price|
+-------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+--------------------+-------------+---------------+------------+-----------------+
|  count|               120|               120|               120|               120|              120|               120|               120|                 120|          120|            120|         120|              120|
|   mean|              60.5|2.0791666666666666| 2.183333333333333| 2.591666666666667|5.483333333333333| 

## Doing OneHot Encoding

In [32]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline
indexer = StringIndexer(inputCol=DF.columns[8], outputCol=DF.columns[8]+'_idx')
df_indexed = indexer.fit(DF).transform(DF)
encoder = OneHotEncoder(inputCol=DF.columns[8]+'_idx', outputCol=DF.columns[8]+"_ohe")
df_encoded = encoder.fit(df_indexed).transform(df_indexed)


indexer = StringIndexer(inputCol=DF.columns[9], outputCol=DF.columns[9]+'_idx')
df_indexed = indexer.fit(df_encoded).transform(df_encoded)
encoder = OneHotEncoder(inputCol=DF.columns[9]+'_idx', outputCol=DF.columns[9]+"_ohe")
df_encoded = encoder.fit(df_indexed).transform(df_indexed)


indexer = StringIndexer(inputCol=DF.columns[10], outputCol=DF.columns[10]+'_idx')
df_indexed = indexer.fit(df_encoded).transform(df_encoded)
encoder = OneHotEncoder(inputCol=DF.columns[10]+'_idx', outputCol=DF.columns[10]+"_ohe")
df_encoded = encoder.fit(df_indexed).transform(df_indexed)

In [33]:
df_encoded.show()

+---+---------+--------+----+------------+--------------+-----------------+--------------------+-------------+---------------+---------------+------+-----------------+-----------------+-------------+-------------+----------------+----------------+
| id|bathrooms|bedrooms|beds|accommodates|minimum_nights|number_of_reviews|review_scores_rating|property_type|      room_type|   neighborhood| price|property_type_idx|property_type_ohe|room_type_idx|room_type_ohe|neighborhood_idx|neighborhood_ohe|
+---+---------+--------+----+------------+--------------+-----------------+--------------------+-------------+---------------+---------------+------+-----------------+-----------------+-------------+-------------+----------------+----------------+
|  1|      1.0|     1.0| 2.0|         3.0|           2.0|             57.0|                74.2|    Apartment|    Shared room|Garden District|146.65|              1.0|    (4,[1],[1.0])|          1.0|(2,[1],[1.0])|             4.0|       (4,[],[])|
|  2|   

In [43]:
df_encoded.columns[1:7]

['bathrooms',
 'bedrooms',
 'beds',
 'accommodates',
 'minimum_nights',
 'number_of_reviews']

In [56]:
from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(
    inputCols=['property_type_ohe','room_type_ohe','neighborhood_ohe','bathrooms','bedrooms','beds','accommodates','minimum_nights','number_of_reviews','review_scores_rating'],  # numeric cols + ohe cols
    outputCol="features"
)
df_encoded=assembler.transform(df_encoded.drop('features'))

In [58]:
ass = VectorAssembler(inputCols=['bathrooms','bedrooms','beds','accommodates','minimum_nights','number_of_reviews','review_scores_rating'],outputCol='features1')
df_encoded = ass.transform(df_encoded.drop('features1'))

In [60]:
df_encoded.show()

+---+---------+--------+----+------------+--------------+-----------------+--------------------+-------------+---------------+---------------+------+-----------------+-----------------+-------------+-------------+----------------+----------------+--------------------+--------------------+
| id|bathrooms|bedrooms|beds|accommodates|minimum_nights|number_of_reviews|review_scores_rating|property_type|      room_type|   neighborhood| price|property_type_idx|property_type_ohe|room_type_idx|room_type_ohe|neighborhood_idx|neighborhood_ohe|            features|           features1|
+---+---------+--------+----+------------+--------------+-----------------+--------------------+-------------+---------------+---------------+------+-----------------+-----------------+-------------+-------------+----------------+----------------+--------------------+--------------------+
|  1|      1.0|     1.0| 2.0|         3.0|           2.0|             57.0|                74.2|    Apartment|    Shared room|Gard

## Scaling input Features

In [67]:
from pyspark.ml.feature import StandardScaler
scale = StandardScaler(inputCol='features',outputCol='feature')
df_encoded=scale.fit(df_encoded.drop('feature')).transform(df_encoded.drop('feature'))

## splitting dataframe

In [68]:
train_df, test_df = df_encoded.randomSplit([0.8, 0.2], seed=42)

In [69]:
from pyspark.ml.regression import LinearRegression
LR = LinearRegression(featuresCol='features',labelCol='price')
lr = LR.fit(train_df)

In [72]:
price_pred=lr.transform(test_df)


In [73]:
price_pred.show()

+---+---------+--------+----+------------+--------------+-----------------+--------------------+-------------+---------------+---------------+------+-----------------+-----------------+-------------+-------------+----------------+----------------+--------------------+--------------------+--------------------+------------------+
| id|bathrooms|bedrooms|beds|accommodates|minimum_nights|number_of_reviews|review_scores_rating|property_type|      room_type|   neighborhood| price|property_type_idx|property_type_ohe|room_type_idx|room_type_ohe|neighborhood_idx|neighborhood_ohe|            features|           features1|             feature|        prediction|
+---+---------+--------+----+------------+--------------+-----------------+--------------------+-------------+---------------+---------------+------+-----------------+-----------------+-------------+-------------+----------------+----------------+--------------------+--------------------+--------------------+------------------+
|100|     

## Model Evaluation

In [74]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction")

mse  = evaluator.setMetricName("mse").evaluate(price_pred)
rmse = evaluator.setMetricName("rmse").evaluate(price_pred)
mae  = evaluator.setMetricName("mae").evaluate(price_pred)
r2   = evaluator.setMetricName("r2").evaluate(price_pred)

print(f"MSE:  {mse}")
print(f"RMSE: {rmse}")
print(f"MAE:  {mae}")
print(f"R2:   {r2}")

MSE:  274.60778916792066
RMSE: 16.571294130752754
MAE:  13.810164708538974
R2:   0.9758631493858202


In [75]:
from pyspark.ml.regression import LinearRegression
LR = LinearRegression(featuresCol='features1',labelCol='price')
lr = LR.fit(train_df)
price_predict=lr.transform(test_df)


In [76]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction")

mse  = evaluator.setMetricName("mse").evaluate(price_predict)
rmse = evaluator.setMetricName("rmse").evaluate(price_predict)
mae  = evaluator.setMetricName("mae").evaluate(price_predict)
r2   = evaluator.setMetricName("r2").evaluate(price_predict)

print(f"MSE:  {mse}")
print(f"RMSE: {rmse}")
print(f"MAE:  {mae}")
print(f"R2:   {r2}")

MSE:  507.5652918828727
RMSE: 22.52920974829949
MAE:  18.179615876181774
R2:   0.9553871808798983
